# Module 8: Storage Theory and Fundamental Bayesian Models

## Learning Objectives
By completing this notebook, you will:
1. Understand the theory of storage and convenience yield
2. Implement nonlinear price-inventory relationships in PyMC
3. Build Bayesian models integrating fundamental covariates (stocks, production, demand)
4. Apply Working's curve to commodity price forecasting
5. Combine time series models with structural economic relationships

## Prerequisites
- Bayesian regression
- Commodity market fundamentals
- PyMC model building

## Estimated Time: 85 minutes

---

In [ ]:
learning_objectives(["Understand the theory of storage and convenience yield", "Implement nonlinear price-inventory relationships in PyMC", "Build Bayesian models integrating fundamental covariates (stocks, production, demand)", "Apply Working's curve to commodity price forecasting", "Combine time series models with structural economic relationships", "Bayesian regression"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az
from scipy import stats

np.random.seed(42)
az.style.use('arviz-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"PyMC version: {pm.__version__}")

## Conceptual Introduction

### The Theory of Storage

**Core Principle:** Commodity prices reflect the **marginal convenience yield** of holding inventory.

**Convenience Yield:** The non-monetary benefit of holding physical inventory:
- Ability to meet unexpected demand
- Avoid stockouts and production shutdowns
- Flexibility in timing sales

### Working's Curve

**Empirical relationship:** Price is a **convex decreasing function** of inventory:

$$P_t = \alpha + \frac{\beta}{S_t} + \epsilon_t$$

Where:
- $P_t$ = Spot price
- $S_t$ = Stocks-to-use ratio (inventory / annual consumption)
- $\alpha$ = Base price (production cost)
- $\beta$ = Scarcity premium

**Intuition:**
- **Low stocks** (S < 15%): High convenience yield → High prices
- **High stocks** (S > 30%): Low convenience yield → Prices near marginal cost
- **Nonlinearity:** Price increase accelerates as stocks decline

### Volatility-Inventory Relationship

**Additional empirical fact:** Price volatility ALSO increases with scarcity:

$$\sigma_t = \sigma_0 + \frac{\gamma}{S_t}$$

**Why?** Low inventory → Higher sensitivity to supply/demand shocks.

### Examples

| Commodity | Critical Stocks-to-Use | Price Behavior |
|-----------|----------------------|----------------|
| Corn | < 15% | Explosive price spikes (2012, 2021) |
| Crude Oil | < 20 days of supply | Backwardation, volatility |
| Natural Gas | < 2 Tcf (US) | Winter price spikes |
| Copper | < 3 weeks | Supply disruption premium |

## Part 1: Working's Curve - Nonlinear Price-Inventory Model

In [ ]:
# Purpose: Generate synthetic corn data with realistic price-inventory relationship
# Key Concept: Convex relationship (low stocks → explosive prices)

n_years = 30

# True parameters (Working's curve)
alpha_true = 3.0  # Base price ($/bushel)
beta_true = 0.5   # Scarcity premium

# Stocks-to-use ratio varies from 0.10 to 0.35 (10% to 35%)
stocks_to_use = np.random.uniform(0.10, 0.35, n_years)

# Expected price (Working's curve)
price_mean = alpha_true + beta_true / stocks_to_use

# Volatility increases with scarcity
sigma_0 = 0.2
gamma_true = 0.02
price_volatility = sigma_0 + gamma_true / stocks_to_use

# Generate observed prices (heteroskedastic noise)
corn_prices = price_mean + price_volatility * np.random.normal(size=n_years)

# Create DataFrame
df = pd.DataFrame({
    'stocks_to_use': stocks_to_use,
    'price': corn_prices,
    'price_mean': price_mean,
    'price_volatility': price_volatility
})

print("Generated data summary:")
print(df.describe())
print(f"\nTrue parameters: α={alpha_true}, β={beta_true}, γ={gamma_true}")

In [ ]:
# Purpose: Visualize Working's curve
# Key Concept: Convex price-inventory relationship

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Price vs Stocks-to-Use
ax = axes[0]

# Scatter observed data
ax.scatter(stocks_to_use, corn_prices, s=80, alpha=0.7, 
          edgecolor='black', linewidth=1.5, label='Observed Prices')

# Plot true Working's curve
s_range = np.linspace(0.10, 0.35, 100)
p_true = alpha_true + beta_true / s_range
ax.plot(s_range, p_true, 'r-', linewidth=3, label="True Working's Curve")

# Highlight critical region
ax.axvspan(0.10, 0.15, alpha=0.2, color='red', label='Critical (< 15%)')
ax.axvspan(0.25, 0.35, alpha=0.2, color='green', label='Comfortable (> 25%)')

ax.set_xlabel('Stocks-to-Use Ratio', fontsize=12)
ax.set_ylabel('Price ($/bushel)', fontsize=12)
ax.set_title("Working's Curve: Price vs Inventory", fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Plot 2: Volatility vs Stocks-to-Use
ax = axes[1]

# Compute realized volatility (residuals)
residuals = corn_prices - price_mean
abs_residuals = np.abs(residuals)

ax.scatter(stocks_to_use, abs_residuals, s=80, alpha=0.7,
          edgecolor='black', linewidth=1.5, label='Absolute Residuals')

# Plot true volatility function
vol_true = sigma_0 + gamma_true / s_range
ax.plot(s_range, vol_true, 'r-', linewidth=3, label='True Volatility Function')

ax.set_xlabel('Stocks-to-Use Ratio', fontsize=12)
ax.set_ylabel('Price Volatility', fontsize=12)
ax.set_title('Volatility Increases with Scarcity', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey observations:")
print("1. Price is CONVEX in inventory (accelerates as stocks decline)")
print("2. Volatility INCREASES with scarcity (heteroskedasticity)")
print("3. Critical stocks (< 15%) associated with explosive prices")

In [ ]:
# Purpose: Build Bayesian Working's curve model
# Key Concept: Nonlinear regression with heteroskedastic errors

with pm.Model() as workings_curve_model:
    # Priors on Working's curve parameters
    alpha = pm.Normal('alpha', mu=3, sigma=1)  # Base price
    beta = pm.HalfNormal('beta', sigma=2)  # Scarcity premium (must be positive)
    
    # Priors on volatility parameters
    sigma_0 = pm.HalfNormal('sigma_0', sigma=0.5)  # Base volatility
    gamma = pm.HalfNormal('gamma', sigma=0.1)  # Volatility scarcity effect
    
    # Expected price (Working's curve)
    price_expected = pm.Deterministic(
        'price_expected',
        alpha + beta / stocks_to_use
    )
    
    # Heteroskedastic volatility (depends on stocks)
    price_sigma = pm.Deterministic(
        'price_sigma',
        sigma_0 + gamma / stocks_to_use
    )
    
    # Likelihood
    pm.Normal(
        'price_obs',
        mu=price_expected,
        sigma=price_sigma,
        observed=corn_prices
    )
    
    # Sample
    trace_workings = pm.sample(
        2000,
        tune=2000,
        target_accept=0.95,
        return_inferencedata=True,
        random_seed=42
    )

print("\nPosterior summary:")
print(az.summary(
    trace_workings,
    var_names=['alpha', 'beta', 'sigma_0', 'gamma'],
    round_to=3
))

print(f"\nTrue values for comparison:")
print(f"α={alpha_true}, β={beta_true}, σ_0={sigma_0}, γ={gamma_true}")

In [ ]:
# Purpose: Visualize posterior predictions
# Key Concept: Model captures nonlinear relationship with uncertainty

# Extract posterior samples
alpha_samples = trace_workings.posterior['alpha'].values.flatten()
beta_samples = trace_workings.posterior['beta'].values.flatten()

# Generate predictions over range of stocks
s_pred = np.linspace(0.10, 0.35, 100)
price_pred_samples = alpha_samples[:, None] + beta_samples[:, None] / s_pred[None, :]

# Compute quantiles
price_pred_mean = price_pred_samples.mean(axis=0)
price_pred_lower = np.percentile(price_pred_samples, 2.5, axis=0)
price_pred_upper = np.percentile(price_pred_samples, 97.5, axis=0)

# Plot
fig, ax = plt.subplots(figsize=(12, 7))

# Observed data
ax.scatter(stocks_to_use, corn_prices, s=100, alpha=0.7, 
          edgecolor='black', linewidth=1.5, label='Observed Prices', zorder=5)

# Posterior mean
ax.plot(s_pred, price_pred_mean, 'b-', linewidth=3, label='Posterior Mean')

# Credible interval
ax.fill_between(s_pred, price_pred_lower, price_pred_upper, 
               alpha=0.3, color='blue', label='95% Credible Interval')

# True curve
p_true_plot = alpha_true + beta_true / s_pred
ax.plot(s_pred, p_true_plot, 'r--', linewidth=2, label='True Curve', alpha=0.7)

# Critical regions
ax.axvspan(0.10, 0.15, alpha=0.15, color='red')
ax.text(0.125, ax.get_ylim()[1]*0.95, 'Critical\nStocks', 
       ha='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Stocks-to-Use Ratio', fontsize=13)
ax.set_ylabel('Price ($/bushel)', fontsize=13)
ax.set_title("Bayesian Working's Curve: Posterior Predictions", fontsize=15, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nModel successfully recovered nonlinear price-inventory relationship")
print("Credible interval captures uncertainty in both curve shape and scatter")

## Exercise 8.1: Price Forecasting with Inventory Projections

**Task:** Use the fitted Working's curve model to forecast prices under three inventory scenarios:

1. **Tight scenario**: Stocks-to-use = 0.12 (below critical)
2. **Baseline scenario**: Stocks-to-use = 0.20 (moderate)
3. **Abundant scenario**: Stocks-to-use = 0.32 (comfortable)

For each scenario:
- Generate posterior predictive price distribution
- Compute mean, median, and 90% credible interval
- Calculate probability of price exceeding $6/bushel

**Expected Output:**
- Tight scenario: High mean price (~$6-7), wide CI, high P(price > 6)
- Baseline: Moderate price (~$4-5), moderate CI
- Abundant: Low price (~$3-4), narrow CI, low P(price > 6)

**Hints:**
<details>
<summary>Hint 1</summary>
Use pm.sample_posterior_predictive() with new stocks_to_use values
</details>

<details>
<summary>Hint 2</summary>
Create a new model context with stocks_to_use as Data container that can be updated
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
# 1. Define three inventory scenarios
# 2. For each, sample from posterior predictive
# 3. Compute summary statistics and probabilities
# 4. Visualize comparison

scenarios = {
    'Tight': 0.12,
    'Baseline': 0.20,
    'Abundant': 0.32
}

# YOUR CODE: Generate predictions for each scenario

your_answer = None  # Dictionary with results

In [ ]:
# AUTO-GRADED TESTS
def test_exercise_8_1():
    assert your_answer is not None, "Implement your solution!"
    assert len(your_answer) == 3, "Need results for 3 scenarios"
    
    # Tight scenario should have higher mean price
    # (Simplified check)
    
    print("Exercise 8.1 passed!")
    print("Successfully generated scenario-based forecasts using fundamentals")

test_exercise_8_1()

## Part 2: Multi-Factor Fundamental Model

### Extending Beyond Inventory

Real commodity prices depend on multiple fundamentals:

$$P_t = \alpha + \frac{\beta}{S_t} + \gamma_1 D_t + \gamma_2 W_t + \gamma_3 E_t + \epsilon_t$$

Where:
- $S_t$ = Stocks-to-use
- $D_t$ = Demand proxy (GDP, livestock inventories, etc.)
- $W_t$ = Weather variables (drought index, temperature)
- $E_t$ = Energy prices (corn-ethanol link)

**Bayesian advantages:**
1. Uncertainty quantification in all coefficients
2. Hierarchical priors on related fundamentals
3. Missing data handling (weather stations, revisions)
4. Regime-dependent coefficients

In [ ]:
# Purpose: Generate multi-factor corn data
# Key Concept: Price depends on inventory, demand, and energy

n_multi = 40

# Fundamental variables
stocks_multi = np.random.uniform(0.12, 0.32, n_multi)
demand_growth = np.random.normal(0.02, 0.01, n_multi)  # Annual demand growth
oil_price = np.random.normal(70, 15, n_multi)  # Crude oil ($/barrel)

# True model
alpha_multi = 2.5
beta_stocks = 0.4
gamma_demand = 100.0  # Positive demand impact
gamma_oil = 0.02      # Positive oil price impact (ethanol link)
sigma_multi = 0.3

# Generate prices
price_multi_mean = (
    alpha_multi + 
    beta_stocks / stocks_multi + 
    gamma_demand * demand_growth + 
    gamma_oil * oil_price
)
price_multi = price_multi_mean + np.random.normal(0, sigma_multi, n_multi)

# Create DataFrame
df_multi = pd.DataFrame({
    'price': price_multi,
    'stocks': stocks_multi,
    'demand_growth': demand_growth,
    'oil_price': oil_price
})

print("Multi-factor data:")
print(df_multi.describe())
print(f"\nTrue coefficients: β_stocks={beta_stocks}, γ_demand={gamma_demand}, γ_oil={gamma_oil}")

In [ ]:
# Purpose: Fit multi-factor fundamental model
# Key Concept: Combine nonlinear (stocks) and linear (demand, oil) effects

with pm.Model() as multifactor_model:
    # Priors
    alpha_mf = pm.Normal('alpha', mu=3, sigma=2)
    beta_mf = pm.HalfNormal('beta_stocks', sigma=1)
    gamma_demand_mf = pm.Normal('gamma_demand', mu=0, sigma=200)  # Wide prior
    gamma_oil_mf = pm.Normal('gamma_oil', mu=0, sigma=0.1)
    sigma_mf = pm.HalfNormal('sigma', sigma=1)
    
    # Expected price
    price_exp_mf = pm.Deterministic(
        'price_expected',
        alpha_mf + 
        beta_mf / stocks_multi + 
        gamma_demand_mf * demand_growth + 
        gamma_oil_mf * oil_price
    )
    
    # Likelihood
    pm.Normal('price_obs', mu=price_exp_mf, sigma=sigma_mf, observed=price_multi)
    
    # Sample
    trace_mf = pm.sample(
        2000,
        tune=2000,
        target_accept=0.95,
        return_inferencedata=True,
        random_seed=42
    )

print("\nMulti-factor model results:")
print(az.summary(
    trace_mf,
    var_names=['alpha', 'beta_stocks', 'gamma_demand', 'gamma_oil'],
    round_to=3
))

In [ ]:
# Purpose: Decompose price into fundamental drivers
# Key Concept: Attribution analysis - what drives price changes?

# Extract posterior means
beta_est = trace_mf.posterior['beta_stocks'].mean().item()
gamma_d_est = trace_mf.posterior['gamma_demand'].mean().item()
gamma_o_est = trace_mf.posterior['gamma_oil'].mean().item()

# Compute contributions
contrib_stocks = beta_est / stocks_multi
contrib_demand = gamma_d_est * demand_growth
contrib_oil = gamma_o_est * oil_price

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Stacked contributions
ax = axes[0]
years = np.arange(n_multi)
ax.bar(years, contrib_stocks, label='Inventory Effect', alpha=0.8)
ax.bar(years, contrib_demand, bottom=contrib_stocks, label='Demand Effect', alpha=0.8)
ax.bar(years, contrib_oil, 
      bottom=contrib_stocks + contrib_demand, 
      label='Oil Price Effect', alpha=0.8)
ax.plot(years, price_multi, 'ro-', linewidth=2, markersize=8, label='Observed Price')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Price ($/bushel)', fontsize=12)
ax.set_title('Price Decomposition: Fundamental Drivers', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Coefficient posteriors
ax = axes[1]
az.plot_forest(
    trace_mf,
    var_names=['beta_stocks', 'gamma_demand', 'gamma_oil'],
    combined=True,
    ax=ax
)
true_vals_mf = [beta_stocks, gamma_demand, gamma_oil]
for i, true_val in enumerate(true_vals_mf):
    ax.axvline(true_val, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax.set_title('Fundamental Coefficients (Red = True)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nAttribution analysis:")
print(f"Average contribution to price:")
print(f"  Inventory: {contrib_stocks.mean():.2f} $/bushel")
print(f"  Demand: {contrib_demand.mean():.2f} $/bushel")
print(f"  Oil: {contrib_oil.mean():.2f} $/bushel")

## Summary

### Key Takeaways

1. **Working's curve captures scarcity premium** - Nonlinear price-inventory relationship

2. **Heteroskedasticity matters** - Volatility increases with scarcity

3. **Multi-factor models** - Combine inventory, demand, and linkage effects

4. **Bayesian uncertainty quantification** - Credible intervals for forecasts and coefficients

5. **Fundamental analysis complements time series** - Structural knowledge improves forecasts

### Practical Applications

| Use Case | Model Type |
|----------|------------|
| Scenario analysis | Working's curve + inventory projections |
| Policy impact | Multi-factor with regime switching |
| Risk management | Volatility-inventory relationship |
| Trading signals | Deviation from fundamental value |

### What's Next

**Combine everything:** Time series (state space, GP) + Fundamentals (Working's curve) + Regime switching

**Real-world deployment:** Production forecasting systems integrating USDA data, weather forecasts, and Bayesian models

---

*"Time series tells you where prices have been. Fundamentals tell you where they should go. Bayesian methods quantify how certain you should be."*